In [0]:
%run "../SetUp/setup" 

### Access Azure Data Lake using Service Principal
**Steps to follow:**
1. Register Azure AD Application/ Service Principal
2. Generate a secret/ password for the application
3. Set spark config with App/ Client Id, Directory/ Tenant Id & Secret
4. Assign role "Storage Blob Data Contributor" to the Data Lake

path,name,size,modificationTime
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-03-21/,2021-03-21/,0,1768733801000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-03-28/,2021-03-28/,0,1768733591000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-04-18/,2021-04-18/,0,1768733721000


In [0]:
%run "../Includes/configs" 

In [0]:
%run "../Includes/comm_func" 

In [0]:
dbutils.widgets.text("p_file_date", "")

In [0]:
v_file_date = dbutils.widgets.get("p_file_date")

**Requirement: Produce a dataset that will serve a dashboard with driver's race results**

In [0]:
#Working on tables: circuits, drivers, constructors, races, results
circuits_df = spark.read.parquet(f"{processed_folder_path}/circuits").withColumnRenamed("location", "circuit_location")

In [0]:
drivers_df = spark.read.parquet(f"{processed_folder_path}/drivers").withColumnRenamed("number", "driver_number").withColumnRenamed("name", "driver_name").withColumnRenamed("nationality", "driver_nationality")

In [0]:
constructors_df = spark.read.parquet(f"{processed_folder_path}/constructors").withColumnRenamed("name", "team")

In [0]:
races_df = spark.read.parquet(f"{processed_folder_path}/races").withColumnRenamed("name", "race_name").withColumnRenamed("race_timestamp", "race_date")

In [0]:
from pyspark.sql.functions import col, to_date, lit

In [0]:
results_df = spark.read.parquet(f"{processed_folder_path}/results").filter(col("file_date") == to_date(lit(v_file_date))).withColumnRenamed("time", "race_time").withColumnRenamed("race_id", "result_race_id")

In [0]:
spark.read.parquet(f"{processed_folder_path}/results") \
    .select("file_date") \
    .distinct() \
    .show(truncate=False)

+----------+
|file_date |
+----------+
|2021-03-21|
|2021-04-18|
|2021-03-28|
+----------+



In [0]:
%sql
SELECT * FROM f1_processed.results
ORDER BY race_id DESC;

result_id,race_id,driver_id,constructor_id,position,position_text,position_order,points,fastest_lap,fastest_lap_time,fastest_lap_speed,ingestion_date,data_source,file_date
24986,1053,830,9,1,1,1,25.0,60,1:17.524,227.9600067138672,2026-01-21T09:58:31.702873Z,Ergast API,2021-04-18
24987,1053,1,131,2,2,2,19.0,60,1:16.702,230.4029998779297,2026-01-21T09:58:31.702873Z,Ergast API,2021-04-18
24988,1053,846,1,3,3,3,15.0,63,1:18.259,225.81900024414062,2026-01-21T09:58:31.702873Z,Ergast API,2021-04-18
24989,1053,844,6,4,4,4,12.0,60,1:18.379,225.47300720214844,2026-01-21T09:58:31.702873Z,Ergast API,2021-04-18
24990,1053,832,6,5,5,5,10.0,60,1:18.490,225.1540069580078,2026-01-21T09:58:31.702873Z,Ergast API,2021-04-18
24991,1053,817,1,6,6,6,8.0,54,1:19.341,222.73899841308594,2026-01-21T09:58:31.702873Z,Ergast API,2021-04-18
24992,1053,842,213,7,7,7,6.0,52,1:18.994,223.71800231933594,2026-01-21T09:58:31.702873Z,Ergast API,2021-04-18
24993,1053,840,117,8,8,8,4.0,59,1:18.782,224.32000732421875,2026-01-21T09:58:31.702873Z,Ergast API,2021-04-18
24994,1053,839,214,9,9,9,2.0,62,1:19.422,222.51199340820312,2026-01-21T09:58:31.702873Z,Ergast API,2021-04-18
24995,1053,4,214,10,10,10,1.0,62,1:19.417,222.5260009765625,2026-01-21T09:58:31.702873Z,Ergast API,2021-04-18


**Join circuits to races**

In [0]:
race_circuits_df = races_df.join(circuits_df, races_df.circuit_id == circuits_df.circuit_id, "inner").select(races_df.race_id, races_df. race_year, races_df.race_name, races_df.race_date, circuits_df.circuit_location)

**Join results to all other dataframes**

In [0]:
race_results_df = results_df.join(race_circuits_df, results_df.result_race_id == race_circuits_df.race_id, "inner").join(drivers_df, results_df.driver_id == drivers_df.driver_id, "inner").join(constructors_df, results_df.constructor_id == constructors_df.constructor_id, "inner")

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
final_df = race_results_df.select("race_id", "race_year", "race_name", "race_date", "circuit_location", "driver_name", "driver_number", "driver_nationality", "team", "grid", "fastest_lap", "race_time", "points", "position").withColumn("created_date", current_timestamp()).withColumn("file_date", lit(v_file_date))

In [0]:
race_results_final_df = final_df.dropDuplicates(
    ["race_id", "driver_name"]
)

In [0]:
race_results_final_df.write \
    .mode("append") \
    .partitionBy("race_id") \
    .parquet(f"{presentation_folder_path}/race_results")

In [0]:
spark.read.parquet(f"{presentation_folder_path}/race_results") \
    .select("file_date") \
    .distinct() \
    .show(truncate=False)

+----------+
|file_date |
+----------+
|2021-03-21|
|2021-04-18|
|2021-03-28|
+----------+



In [0]:
#display(final_df.filter("race_year == 2020 and race_name == 'Abu Dhabi Grand Prix'").orderBy(final_df.points.desc()))

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:134)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:477)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.data

In [0]:
#final_df.write.mode("overwrite").parquet(f"{presentation_folder_path}/race_results")

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:134)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:477)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.data

In [0]:
%sql
--DROP TABLE IF EXISTS f1_presentation.race_results;

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:134)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:477)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.data

In [0]:
%sql
CREATE TABLE IF NOT EXISTS f1_presentation.race_results (
  race_id INT,
  race_year INT,
  race_name STRING,
  race_date TIMESTAMP,
  circuit_location STRING,
  driver_name STRING,
  driver_number INT,
  driver_nationality STRING,
  team STRING,
  grid INT,
  fastest_lap INT,
  race_time STRING,
  points DOUBLE,
  position INT,
  created_date TIMESTAMP,
  file_date DATE
)
USING DELTA
PARTITIONED BY (race_id);

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:134)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:477)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.data

In [0]:
merge_into_table(
    final_df,
    "f1_presentation",
    "race_results",
    """
    target.race_id = source.race_id
    AND target.driver_name = source.driver_name
    """
)

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:134)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:477)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.data

In [0]:
%sql
SELECT race_id, COUNT(*) 
FROM f1_presentation.race_results
GROUP BY race_id
ORDER BY race_id DESC;

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:134)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:477)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.data

In [0]:
final_df.count()

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:134)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:477)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.data

In [0]:
%sql
SELECT * FROM f1_presentation.race_results
ORDER BY race_id DESC;

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:134)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:477)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.data